# Demo — AgentCore Gateway Governed Tool Access

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/bedrock-companion/blob/main/day1/demos/demo-gateway-tool-access/demo-gateway-tool-access.ipynb)

This notebook is a follow-along demo.

Day 1 — Block 2: Core Execution (Runtime and Gateway)

Demonstrates how AgentCore Gateway exposes approved tools through MCP or HTTP
targets, and why preventing direct Runtime bypass is critical.

Prerequisites: AWS credentials configured.

In [ ]:
# Install required dependencies
!pip install boto3 --quiet

In [ ]:
import json
import os
import uuid

try:
    import boto3
    HAS_BOTO3 = True
except ImportError:
    HAS_BOTO3 = False

# --- Configuration ---
GATEWAY_NAME = os.environ.get("GATEWAY_NAME", "ProdGateway")
GATEWAY_ROLE_ARN = os.environ.get(
    "GATEWAY_ROLE_ARN",
    "arn:aws:iam::123456789012:role/AgentCoreGatewayRole",
)

if HAS_BOTO3:
    control = boto3.client("bedrock-agentcore-control")

def print_section(title: str):
    print(f"\n{'=' * 60}")
    print(f"  {title}")
    print(f"{'=' * 60}")

def create_gateway():
    """Create an AgentCore Gateway with MCP protocol support.
    This is the boto3 pattern for create_gateway."""
    print_section("Creating AgentCore Gateway")

    params = {
        "name": GATEWAY_NAME,
        "protocolType": "MCP",
        "protocolConfiguration": {
            "mcp": {
                "supportedVersions": ["2024-11-05", "2025-03-26"],
                "instructions": "Gateway for production agent tools",
                "sessionConfiguration": {
                    "sessionTimeoutInSeconds": 3600,
                },
            }
        },
        "authorizerType": "AWS_IAM",
        "roleArn": GATEWAY_ROLE_ARN,
        "clientToken": str(uuid.uuid4()),
    }

    print("  API: bedrock-agentcore-control.create_gateway")
    print(f"  Parameters:\n{json.dumps(params, indent=4)}")

    # In production, uncomment to actually deploy:
    # response = control.create_gateway(**params)
    # print(f"  Created: {response['gatewayArn']}")

    print("\n  [Simulated] Gateway created successfully.")
    return f"arn:aws:bedrock-agentcore:us-east-1:123456789012:gateway/{GATEWAY_NAME}"

def gateway_mcp_translation():
    """Show how Gateway translates tool calls to MCP JSON-RPC.

    AgentCore Gateway converts REST, Lambda, and Smithy models into
    standardized MCP tools that agents can invoke."""
    print_section("Gateway MCP Protocol Translation")

    # Agent requests a tool in natural format
    agent_request = {
        "tool": "get_order_status",
        "parameters": {"order_id": "ORD-555"},
    }
    print(f"  Agent request: {json.dumps(agent_request)}")

    # Gateway translates to MCP JSON-RPC
    mcp_request = {
        "jsonrpc": "2.0",
        "id": str(uuid.uuid4())[:8],
        "method": "tools/call",
        "params": {
            "name": "get_order_status",
            "arguments": {"order_id": "ORD-555"},
        },
    }
    print(f"\n  Gateway translates to MCP:\n  {json.dumps(mcp_request, indent=2)}")

    # Target responds
    mcp_response = {
        "jsonrpc": "2.0",
        "id": mcp_request["id"],
        "result": {
            "content": [{"type": "text", "text": "Order ORD-555 is Shipped."}],
            "isError": False,
        },
    }
    print(f"\n  Target response:\n  {json.dumps(mcp_response, indent=2)}")

    # Gateway translates back
    agent_response = {
        "tool": "get_order_status",
        "status": "success",
        "output": "Order ORD-555 is Shipped.",
    }
    print(f"\n  Gateway translates back to agent:\n  {json.dumps(agent_response, indent=2)}")

def gateway_vs_api_gateway():
    """Distinguish AgentCore Gateway from Amazon API Gateway.

    This is a key learning outcome: students must understand where each
    belongs in an enterprise architecture."""
    print_section("AgentCore Gateway vs. Amazon API Gateway")

    comparison = {
        "AgentCore Gateway": {
            "role": "Agent-facing tool routing",
            "protocol": "MCP (Model Context Protocol)",
            "functions": [
                "Tool discovery and invocation",
                "Protocol translation (REST→MCP, Lambda→MCP)",
                "Tool allowlisting and governance",
                "Agent-to-agent communication (A2A)",
            ],
        },
        "Amazon API Gateway": {
            "role": "Client-facing API edge",
            "protocol": "HTTP/REST/WebSocket",
            "functions": [
                "Client authentication (Cognito, Okta, etc.)",
                "Throttling and rate limiting",
                "WAF integration and DDoS protection",
                "Request/response transformation",
                "API versioning and canary deployment",
            ],
        },
    }
    print(f"  {json.dumps(comparison, indent=2)}")
    print("\n  Typical enterprise flow:")
    print("  Client → API Gateway (auth, throttle) → Backend → AgentCore Runtime → AgentCore Gateway → Tools")

def direct_bypass_prevention():
    """Demonstrate why preventing direct Runtime invocation is critical.

    The Runtime endpoint must only accept requests from the Gateway role.
    Direct client invocation bypasses tool governance and authorization."""
    print_section("Direct Bypass Prevention")

    print("  Runtime endpoint resource policy:")
    print("  - ALLOW: bedrock-agentcore:InvokeAgentRuntime")
    print("  - CONDITION: aws:PrincipalArn = Gateway execution role")
    print("  - DENY:  All other principals")
    print()
    print("  Without this policy:")
    print("  - Client could invoke Runtime directly, skipping Gateway governance")
    print("  - Tool allowlists, rate limits, and protocol validation are bypassed")
    print("  - Audit trail loses the Gateway hop — harder to trace tool access")
    print()
    print("  IAM condition to enforce Gateway-only access:")
    condition = {
        "Sid": "EnforceGatewayAccess",
        "Effect": "Deny",
        "Action": "bedrock-agentcore:InvokeAgentRuntime",
        "Resource": "*",
        "Condition": {
            "StringNotEquals": {
                "aws:PrincipalArn": GATEWAY_ROLE_ARN,
            },
        },
    }
    print(f"  {json.dumps(condition, indent=2)}")

def main():
    print("AgentCore Gateway — Governed Tool Access Demo")
    print(f"Gateway: {GATEWAY_NAME}\n")

    create_gateway()
    gateway_mcp_translation()
    gateway_vs_api_gateway()
    direct_bypass_prevention()

    print_section("Key Takeaways")
    print("  1. Gateway is agent-facing and tool-aware — it does not replace API Gateway")
    print("  2. MCP protocol standardizes how agents discover and invoke tools")
    print("  3. Gateway translates REST, Lambda, and Smithy into MCP-compatible tools")
    print("  4. Prevent direct Runtime bypass with IAM conditions on the endpoint")
    print("  5. Downstream services must still re-verify authorization per request")

if __name__ == "__main__":
    main()
